## Практическая работа №3. Визуализация графов

**Цель:** Изучить методики визуализации графовых структур, освоить Plotly и Pyvis, сформулировать признаки мошенничеств с графическим обоснованием.

**Задачи:**
1. Загрузить набор данных FinFraud (транзакции МДП)
2. Выполнить анализ с помощью визуализации графов
3. Выявить 2 мошеннические схемы с иллюстрациями

### 1. Подключение библиотек

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import plotly.graph_objects as go
from pyvis.network import Network
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

### 2. Загрузка и предобработка данных

Набор данных — транзакции мобильных денежных переводов (МДП) из проекта MASSIF. Поля: метка (F- = мошенничество), отправитель, получатель, сумма, тип транзакции, роли участников.

In [2]:
cols = ['label', 'sender_id', 'receiver_id', 'sender_acc', 'receiver_acc', 'amount', 'tx_type', 'status',
        'bal_s_before', 'bal_s_after', 'bal_r_before', 'bal_r_after', 'u12', 'u13', 'u14', 'u15',
        'ts_sender', 'ts_receiver', 'acc_sender', 'u19', 'u20', 'tx_id', 'ts', 'sender_type', 'receiver_type']

df = pd.read_csv('csv/FinFraud_Labelled.csv', sep='|', header=None, names=cols, on_bad_lines='skip', low_memory=False)
df = df[df['label'].notna()].copy()
df['amount'] = pd.to_numeric(df['amount'], errors='coerce')
df['is_fraud'] = df['label'].astype(str).str.startswith('F-')

print(f"Всего транзакций: {len(df):,}")
print(f"Мошеннических: {df['is_fraud'].sum():,} ({100*df['is_fraud'].mean():.1f}%)")
print(f"\nТипы мошенничества: {df[df['is_fraud']]['label'].unique().tolist()}")
df.head(10)

Всего транзакций: 54,848
Мошеннических: 717 (1.3%)

Типы мошенничества: ['F-Mule-With']


,label,sender_id,receiver_id,sender_acc,receiver_acc,amount,tx_type,status,bal_s_before,bal_s_after,...,ts_sender,ts_receiver,acc_sender,u19,u20,tx_id,ts,sender_type,receiver_type,is_fraud
1,N-RegC2C,PN_EU_3_4,PN_EU_0_883,EUAcc3_4,EUAcc0_883,68897.74,Ind,SU,1.000000e+08,9.993041e+07,...,01/06/2011 00:09:01,01/06/2011 00:09:01,EUAcc3_4,NaN,NaN,C2C201161.099,01/06/2011 00:09:01,EU,EU,False
2,N-RegC2C,PN_EU_1_139,PN_EU_0_754,EUAcc1_139,EUAcc0_754,68945.47,Ind,SU,1.000000e+08,9.993037e+07,...,01/06/2011 00:15:23,01/06/2011 00:15:23,EUAcc1_139,NaN,NaN,C2C201161.01515,01/06/2011 00:15:23,EU,EU,False
3,N-RegDep,PN_Ret2,PN_EU_3_17,RAcc2,EUAcc3_17,9715.41,Dt,SU,1.000000e+09,9.999903e+08,...,01/06/2011 00:22:07,01/06/2011 00:22:07,RAcc2,NaN,NaN,Dp201161.02222,01/06/2011 00:22:07,RET,EU,False
4,N-RegDep,PN_Ret1,PN_EU_0_266,RAcc1,EUAcc0_266,79303.74,Dt,SU,1.000000e+09,9.999207e+08,...,01/06/2011 00:22:35,01/06/2011 00:22:35,RAcc1,NaN,NaN,Dp201161.02222,01/06/2011 00:22:35,RET,EU,False
5,N_Reg_RC,PN_EU_0_905,operator,EUAcc0_905,A0,929.92,ArRC,SU,1.000000e+08,9.999907e+07,...,01/06/2011 00:29:56,01/06/2011 00:29:56,EUAcc0_905,NaN,NaN,Rc201161.02929,01/06/2011 00:29:56,EU,operator,False
6,N-RegDep,PN_Ret5,PN_EU_2_141,RAcc5,EUAcc2_141,27711.48,Dt,SU,1.000000e+09,9.999723e+08,...,01/06/2011 00:42:37,01/06/2011 00:42:37,RAcc5,NaN,NaN,Dp201161.04242,01/06/2011 00:42:37,RET,EU,False
7,N_RegWith,PN_EU_1_46,PN_Ret5,EUAcc1_46,RAcc5,407023.05,Wl,SU,1.000000e+08,9.958891e+07,...,01/06/2011 00:52:44,01/06/2011 00:52:44,EUAcc1_46,NaN,NaN,Wt201161.05252,01/06/2011 00:52:44,EU,RET,False
8,N_Reg_RC,PN_EU_1_20,operator,EUAcc1_20,A0,2449.60,ArRC,SU,1.000000e+08,9.999755e+07,...,01/06/2011 00:59:03,01/06/2011 00:59:03,EUAcc1_20,NaN,NaN,Rc201161.05959,01/06/2011 00:59:03,EU,operator,False
9,N_Reg_RC,PN_EU_0_801,operator,EUAcc0_801,A0,2176.00,ArRC,SU,1.000000e+08,9.999782e+07,...,01/06/2011 01:06:55,01/06/2011 01:06:55,EUAcc0_801,NaN,NaN,Rc201161.166,01/06/2011 01:06:55,EU,operator,False
10,N_Reg_RC,PN_EU_0_518,operator,EUAcc0_518,A0,5049.15,ArRC,SU,1.000000e+08,9.999495e+07,...,01/06/2011 01:10:36,01/06/2011 01:10:36,EUAcc0_518,NaN,NaN,Rc201161.11010,01/06/2011 01:10:36,EU,operator,False


### 3. Разведочный анализ мошеннических транзакций

In [3]:
fraud = df[df['is_fraud']].copy()
print("Характеристики мошеннических транзакций:")
print(f"  - Тип транзакции: {fraud['tx_type'].value_counts().to_dict()}")
print(f"  - Направление: {fraud.groupby(['sender_type','receiver_type']).size().to_dict()}")
print(f"\nУникальных отправителей (счётов): {fraud['sender_id'].nunique()}")
print(f"Уникальных получателей (агентов): {fraud['receiver_id'].nunique()}")

# Топ скомпрометированных аккаунтов
print("\nТоп-5 скомпрометированных аккаунтов (отправителей):")
fraud['sender_id'].value_counts().head()

Характеристики мошеннических транзакций:
  - Тип транзакции: {'Wl': 717}
  - Направление: {('EU', 'RET'): 717}

Уникальных отправителей (счётов): 4
Уникальных получателей (агентов): 6

Топ-5 скомпрометированных аккаунтов (отправителей):


sender_id
PN_EU_0_955     197
PN_EU_1_328     183
PN_EU_0_1045    170
PN_EU_0_260     167
Name: count, dtype: int64

In [4]:
# Сравнение с легитимными транзакциями
legit = df[~df['is_fraud']]
print("Легитимные транзакции — типы:", legit['tx_type'].value_counts().head().to_dict())
print("\nМошеннические — только снятие (Wl) через агентов: EU -> RET")
print("Паттерн: скомпрометированные аккаунты пользователей выводят средства через агентов.")

Легитимные транзакции — типы: {'ArRC': 28312, 'Dt': 12867, 'Ind': 8205, 'Wl': 4304, 'Merchant': 443}

Мошеннические — только снятие (Wl) через агентов: EU -> RET
Паттерн: скомпрометированные аккаунты пользователей выводят средства через агентов.


### 4. Построение графа транзакций

Граф: узлы — участники (отправители/получатели), рёбра — транзакции. Вес ребра — сумма или количество транзакций.

In [5]:
def build_transaction_graph(transactions, weight_by='count'):
    """Строит ориентированный граф из транзакций."""
    G = nx.DiGraph()
    for _, row in transactions.iterrows():
        s, r = str(row['sender_id']), str(row['receiver_id'])
        if s == r:
            continue
        amt = float(row['amount']) if pd.notna(row['amount']) else 0
        if G.has_edge(s, r):
            G[s][r]['weight'] += amt if weight_by == 'amount' else 1
            G[s][r]['count'] = G[s][r].get('count', 1) + 1
        else:
            G.add_edge(s, r, weight=amt if weight_by == 'amount' else 1, count=1)
    return G

G_fraud = build_transaction_graph(fraud, weight_by='amount')
G_legit_sample = build_transaction_graph(legit.sample(min(5000, len(legit)), random_state=42), weight_by='amount')

print(f"Граф мошенничества: {G_fraud.number_of_nodes()} узлов, {G_fraud.number_of_edges()} рёбер")
print(f"Граф легитимных (выборка): {G_legit_sample.number_of_nodes()} узлов, {G_legit_sample.number_of_edges()} рёбер")

Граф мошенничества: 10 узлов, 24 рёбер
Граф легитимных (выборка): 1728 узлов, 2926 рёбер


### 5. Визуализация графа мошенничества в Plotly

In [6]:
pos = nx.spring_layout(G_fraud, k=1.5, iterations=50, seed=42)

edge_x, edge_y = [], []
edge_weights = []
for u, v, d in G_fraud.edges(data=True):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])
    edge_weights.append(d.get('weight', 1))

node_x = [pos[n][0] for n in G_fraud.nodes()]
node_y = [pos[n][1] for n in G_fraud.nodes()]

# Цвет узлов: EU = скомпрометированные, RET = агенты
node_colors = ['#e74c3c' if 'EU' in str(n) or 'PN_EU' in str(n) else '#3498db' for n in G_fraud.nodes()]
node_sizes = [20 + 2 * G_fraud.out_degree(n) for n in G_fraud.nodes()]

edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=0.5, color='#888'), hoverinfo='none', mode='lines')
node_trace = go.Scatter(x=node_x, y=node_y, mode='markers+text', marker=dict(size=node_sizes, color=node_colors, line=dict(width=1)),
                        text=[str(n)[:15] for n in G_fraud.nodes()], textposition="top center", textfont=dict(size=8))

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(title='Граф мошеннических транзакций (красные — счёта пользователей, синие — агенты)',
                 showlegend=False, hovermode='closest', xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                 yaxis=dict(showgrid=False, zeroline=False, showticklabels=False), height=500)
fig.show()

### 6. Интерактивная визуализация в Pyvis

In [8]:
net = Network(height='500px', width='100%', directed=True, notebook=True)
net.from_nx(G_fraud)

for node in net.nodes:
    nid = node['id']
    node['title'] = f"{nid}\nИсходящих: {G_fraud.out_degree(nid)}"
    if 'EU' in str(nid) or 'PN_EU' in str(nid):
        node['color'] = '#e74c3c'
        node['size'] = 15 + G_fraud.out_degree(nid) * 2
    else:
        node['color'] = '#3498db'
        node['size'] = 20 + G_fraud.in_degree(nid)

net.show('fraud_graph.html')
print("Граф сохранён в fraud_graph.html")

fraud_graph.html
Граф сохранён в fraud_graph.html


### 7. Выявление двух мошеннических схем

В датасете представлены два типа мошенничества (по описанию): **кража мобильного телефона** и **заражение ботнетом**. Оба проявляются как F-Mule-With (вывод через мулов). Различия — в структуре графа:

- **Кража телефона:** один скомпрометированный счёт → много выводов разным агентам (паттерн «звезда»)
- **Ботнет:** много скомпрометированных счетов → концентрация на нескольких агентах (распределённая сеть)

In [9]:
# Анализ: счета с высокой исходящей степенью (много выводов) — возможная кража
out_deg = dict(G_fraud.out_degree())
sender_counts = fraud.groupby('sender_id').size()

# Схема 1: «Звезда» — топ счёт по количеству транзакций (кража одного телефона)
top_sender = sender_counts.idxmax()
scheme1_txs = fraud[fraud['sender_id'] == top_sender]
G_scheme1 = build_transaction_graph(scheme1_txs, weight_by='amount')

# Схема 2: Агенты с высокой входящей степенью (много счётов выводят через них — ботнет)
in_deg = dict(G_fraud.in_degree())
top_agents = sorted(in_deg.items(), key=lambda x: -x[1])[:3]
top_agent_ids = [a[0] for a in top_agents]
scheme2_txs = fraud[fraud['receiver_id'].isin(top_agent_ids)]
G_scheme2 = build_transaction_graph(scheme2_txs, weight_by='amount')

print("Схема 1 (кража телефона): один счёт", top_sender, "->", len(scheme1_txs), "транзакций")
print("Схема 2 (ботнет):", len(scheme2_txs), "транзакций через топ-агентов", top_agent_ids)

Схема 1 (кража телефона): один счёт PN_EU_0_955 -> 197 транзакций
Схема 2 (ботнет): 359 транзакций через топ-агентов ['PN_Ret2', 'PN_Ret6', 'PN_Ret1']


#### Схема 1: Кража мобильного телефона (паттерн «звезда»)

Один скомпрометированный счёт совершает множество выводов через разных агентов. Граф имеет характерную «звезду»: один центральный узел (EU) с множеством исходящих рёбер к агентам (RET).

In [10]:
pos1 = nx.spring_layout(G_scheme1, k=2, seed=42)
edge_x1, edge_y1 = [], []
for u, v in G_scheme1.edges():
    edge_x1.extend([pos1[u][0], pos1[v][0], None])
    edge_y1.extend([pos1[u][1], pos1[v][1], None])

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=edge_x1, y=edge_y1, line=dict(width=2, color='#888'), mode='lines'))

nodes = list(G_scheme1.nodes())
colors = ['#e74c3c' if 'EU' in str(n) else '#3498db' for n in nodes]
sizes = [40 if 'EU' in str(n) else 25 for n in nodes]
fig1.add_trace(go.Scatter(x=[pos1[n][0] for n in nodes], y=[pos1[n][1] for n in nodes],
                         mode='markers+text', marker=dict(size=sizes, color=colors),
                         text=nodes, textposition="top center"))
fig1.update_layout(title='Схема 1: Кража телефона — один счёт (красный) → много агентов (синие)',
                  showlegend=False, xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False), height=400)
fig1.show()

#### Схема 2: Мобильный ботнет (распределённая сеть)

Множество скомпрометированных устройств выводят средства через ограниченное число агентов. Граф: много узлов EU сходятся к нескольким узлам RET — концентрация входящих потоков.

In [11]:
pos2 = nx.spring_layout(G_scheme2, k=1.2, iterations=80, seed=42)
edge_x2, edge_y2 = [], []
for u, v in G_scheme2.edges():
    edge_x2.extend([pos2[u][0], pos2[v][0], None])
    edge_y2.extend([pos2[u][1], pos2[v][1], None])

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=edge_x2, y=edge_y2, line=dict(width=0.8, color='#888'), mode='lines'))

nodes2 = list(G_scheme2.nodes())
colors2 = ['#e74c3c' if 'EU' in str(n) or 'PN_EU' in str(n) else '#3498db' for n in nodes2]
sizes2 = [15 + G_scheme2.out_degree(n) if 'EU' in str(n) else 25 + G_scheme2.in_degree(n) for n in nodes2]
fig2.add_trace(go.Scatter(x=[pos2[n][0] for n in nodes2], y=[pos2[n][1] for n in nodes2],
                         mode='markers+text', marker=dict(size=sizes2, color=colors2),
                         text=[str(n)[:12] for n in nodes2], textposition="top center", textfont=dict(size=8)))
fig2.update_layout(title='Схема 2: Ботнет — много счетов (красные) → несколько агентов (синие)',
                  showlegend=False, xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                  yaxis=dict(showgrid=False, zeroline=False, showticklabels=False), height=450)
fig2.show()

### 8. Сводная статистика и признаки мошенничества

In [12]:
# Количественные признаки
print("Признаки мошенничества (подтверждённые графом):")
print("1. Тип транзакции: только Wl (снятие) — 100% мошеннических")
print("2. Направление: EU → RET (пользователь → агент)")
print("3. Структура графа: двудольный — счета пользователей слева, агенты справа")
print("4. Схема 1: высокая исходящая степень одного узла (звезда)")
print("5. Схема 2: высокая входящая степень агентов (воронка)")

Признаки мошенничества (подтверждённые графом):
1. Тип транзакции: только Wl (снятие) — 100% мошеннических
2. Направление: EU → RET (пользователь → агент)
3. Структура графа: двудольный — счета пользователей слева, агенты справа
4. Схема 1: высокая исходящая степень одного узла (звезда)
5. Схема 2: высокая входящая степень агентов (воронка)
